In [1]:
import requests

URL = "https://books.toscrape.com/catalogue/page-1.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (ton.email@exemple.fr)"}

r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"            # <- a ne pas oublier, on voit pourquoi juste apres

print("status   :", r.status_code)
print("longueur :", len(r.text), "caracteres")
print("debut    :", r.text[:60])

status   : 200
longueur : 50449 caracteres
debut    : 

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [2]:
r = requests.get(URL, headers=ENTETES, timeout=10)

print("encodage devine par requests :", r.encoding)
print("encodage reel de la page     :", r.apparent_encoding)
print("le prix :", r.text[r.text.find("price_color") : r.text.find("price_color") + 40])

encodage devine par requests : ISO-8859-1
encodage reel de la page     : utf-8
le prix : price_color">Â£51.77</p>
    

<p class=


In [3]:
print("le prix est present :", "£51.77" in r.text)
print("le titre est present:", "A Light in the Attic" in r.text)

le prix est present : True
le titre est present: True


In [4]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

fiches = soup.select("article.product_pod")
print("fiches trouvees :", len(fiches))

fiches trouvees : 20


In [5]:
fiche = fiches[0]

# le CONTENU d'une balise
prix = fiche.select_one("p.price_color").get_text(strip=True)

# un ATTRIBUT
lien_h3 = fiche.select_one("h3 a")
titre = lien_h3["title"]
href = lien_h3["href"]

print("prix  :", prix)
print("titre :", titre)
print("href  :", href)

prix  : Â£51.77
titre : A Light in the Attic
href  : a-light-in-the-attic_1000/index.html


In [6]:
classes = fiche.select_one("p.star-rating")["class"]
print("classes :", classes)

classes : ['star-rating', 'Three']


In [7]:
NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

note = NOTES[classes[1]]
print("note :", note)

note : 3


In [8]:
from urllib.parse import urljoin

print(urljoin(URL, href))

https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


In [9]:
def lire_fiche(fiche, url_page):
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]
    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"]),
    }


print(lire_fiche(fiches[0], URL))

{'titre': 'A Light in the Attic', 'prix': 'Â£51.77', 'dispo': 'In stock', 'note': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


In [10]:
def lire_fiche(fiche, url_page):
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]
    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"]),
    }


print(lire_fiche(fiches[0], URL))

{'titre': 'A Light in the Attic', 'prix': 'Â£51.77', 'dispo': 'In stock', 'note': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


In [11]:
suivant = soup.select_one("li.next a")
print(suivant["href"] if suivant else "pas de page suivante")

page-2.html


In [12]:
import time

def collecter_toutes_les_pages(url_depart, pages_max=3):
    url = url_depart
    livres = []
    page = 0

    while url and page < pages_max:
        page += 1
        r = requests.get(url, headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        if r.status_code != 200:
            print(f"  page {page} : status {r.status_code}, on s'arrete")
            break

        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres += [lire_fiche(f, url) for f in fiches]
        print(f"  page {page} : {len(fiches)} livres  (total {len(livres)})")

        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

        time.sleep(1)

    return livres


livres = collecter_toutes_les_pages("https://books.toscrape.com/catalogue/page-1.html")
print("collecte :", len(livres), "livres")

  page 1 : 20 livres  (total 20)
  page 2 : 20 livres  (total 40)
  page 3 : 20 livres  (total 60)
collecte : 60 livres


In [13]:
def lire_detail(url):
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    if r.status_code != 200:
        return {"categorie": None, "stock": None}

    soup = BeautifulSoup(r.text, "html.parser")

    # le tableau : une entree par ligne, le th en cle et le td en valeur
    infos = {}
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        val = ligne.select_one("td")
        if cle and val:
            infos[cle.get_text(strip=True)] = val.get_text(strip=True)

    # la categorie : troisieme element du fil d'Ariane
    miettes = soup.select("ul.breadcrumb li a")
    categorie = miettes[-1].get_text(strip=True) if miettes else None

    return {
        "categorie": categorie,
        "stock": infos.get("Availability"),
        "upc": infos.get("UPC"),
    }


url_test = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
print(lire_detail(url_test))

{'categorie': 'Poetry', 'stock': 'In stock (22 available)', 'upc': 'a897fe39b1053632'}


In [14]:
import re

def nombre_en_stock(texte):
    if not texte:
        return None
    trouve = re.search(r"\((\d+) available\)", texte)
    return int(trouve.group(1)) if trouve else 0


print(nombre_en_stock("In stock (22 available)"))
print(nombre_en_stock("Out of stock"))
print(nombre_en_stock(None))

22
0
None


| Donnée | Sélecteur / Logique | Contenu ou Attribut |
| :--- | :--- | :--- |
| **Titre du livre** | `h3 > a` | Attribut `title` |
| **Prix** | `p.price_color` | Texte (converti en numérique) |
| **Disponibilité** | `p.instock.availability` | Texte (nettoyé) |
| **Note (étoiles)** | `p.star-rating` | Attribut `class` |
| **URL de la fiche** | `h3 > a` | Attribut `href` |
| **Bloc répété (liste)** | `article.product_pod` | Élément conteneur de chaque livre |
| **Lien page suivante** | `li.next > a` | Attribut `href` |
| **Nombre total annoncé** | Sélecteur de pagination | Nombre total de livres dans la catégorie |

In [15]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

# 1. Requête sur la première page de la catégorie
url = "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html"
headers = {"User-Agent": "bootcamp-data/1.0"}

response = requests.get(url, headers=headers, timeout=10)
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

# 2. Extraction des livres de la page
data = []
articles = soup.find_all("article", class_="product_pod")

for article in articles:
  titre = article.find("h3").find("a")["title"]
  prix = article.find("p", class_="price_color").text
  dispo = article.find("p", class_="instock availability").text.strip()

  # La note est dans les classes de la balise p (ex: ['star-rating', 'Three'])
  classes_note = article.find("p", class_="star-rating")["class"]
  note = classes_note[1]  # Récupère la deuxième classe (ex: 'Three')

  url_fiche = article.find("h3").find("a")["href"]

  data.append({
      "titre": titre,
      "prix": prix,
      "dispo": dispo,
      "note": note,
      "url": url_fiche,
  })

# 3. Création et affichage du DataFrame
df = pd.DataFrame(data)
print(df.head())

                                             titre    prix     dispo  note  \
0                                    Sharp Objects  £47.82  In stock  Four   
1                             In a Dark, Dark Wood  £19.63  In stock   One   
2                              The Past Never Ends  £56.50  In stock  Four   
3                                 A Murder in Time  £16.64  In stock   One   
4  The Murder of Roger Ackroyd (Hercule Poirot #4)  £44.10  In stock  Four   

                                                 url  
0              ../../../sharp-objects_997/index.html  
1        ../../../in-a-dark-dark-wood_963/index.html  
2        ../../../the-past-never-ends_942/index.html  
3           ../../../a-murder-in-time_877/index.html  
4  ../../../the-murder-of-roger-ackroyd-hercule-p...  


In [17]:
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html"
headers = {"User-Agent": "bootcamp-data/1.0"}

data = []
page_count = 1

while url:
    print(f"Collecte de la page {page_count}...")
    response = requests.get(url, headers=headers, timeout=10)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")
    articles = soup.find_all("article", class_="product_pod")

    for article in articles:
        titre = article.find("h3").find("a")["title"]
        prix = article.find("p", class_="price_color").text
        dispo = article.find("p", class_="instock availability").text.strip()

        classes_note = article.find("p", class_="star-rating")["class"]
        note = classes_note[1]

        url_fiche = article.find("h3").find("a")["href"]

        # ⚠️ INDENTATION IMPORTANTE : Le append doit être ici, dans la boucle for !
        data.append({
            "titre": titre,
            "prix": prix,
            "dispo": dispo,
            "note": note,
            "url": url_fiche,
        })

    # Recherche du lien "page suivante" (li.next)
    suivant = soup.select_one("li.next > a")

    if suivant:
        href_suivant = suivant["href"]
        url = urljoin(url, href_suivant)
        page_count += 1
        time.sleep(1)
    else:
        print("Fin de la pagination.")
        break

# Création du DataFrame global
df_catalogue = pd.DataFrame(data)
print(f"Nombre total de livres récupérés : {len(df_catalogue)}")

Collecte de la page 1...
Collecte de la page 2...
Fin de la pagination.
Nombre total de livres récupérés : 32


In [18]:
import re
import time
from urllib.parse import urljoin
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Liste pour stocker les données enrichies de niveau 2
data_niveau_2 = []
total_livres = len(df_catalogue)

print("Début de la collecte du second niveau (fiches produits)...")

for index, row in df_catalogue.iterrows():
  # Reconstruction de l'URL absolue du livre à partir de l'URL relative
  # (les URL dans df_catalogue remontent via ../../../)
  base_url = (
      "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html"
  )
  url_livre = urljoin(base_url, row["url"])

  # Affichage de l'avancement
  print(
      f"[{index + 1}/{total_livres}] Scraping de : {row['titre'][:30]}..."
  )

  response = requests.get(url_livre, headers={"User-Agent": "bootcamp-data/1.0"})
  response.encoding = "utf-8"
  soup_livre = BeautifulSoup(response.text, "html.parser")

  # 1. Extraction de la catégorie (dans le fil d'ariane : ul.breadcrumb > li:nth-child(3) > a)
  breadcrumb = soup_livre.select("ul.breadcrumb li")
  categorie = breadcrumb[2].text.strip() if len(breadcrumb) > 2 else "Inconnue"

  # 2. Extraction du stock chiffré (ex: "In stock (22 available)")
  dispo_texte = (
      soup_livre.find("p", class_="instock availability").text.strip()
  )
  # Utilisation d'une expression régulière pour extraire uniquement les chiffres du stock
  match_stock = re.search(r"\((\d+)\s+available\)", dispo_texte)
  stock = int(match_stock.group(1)) if match_stock else 0

  # 3. Extraction de l'UPC (situé dans le tableau de la fiche produit : th == 'UPC' + td)
  upc = None
  table_rows = soup_livre.find_all("tr")
  for tr in table_rows:
    if tr.find("th") and tr.find("th").text == "UPC":
      upc = tr.find("td").text.strip()
      break

  # Nettoyage et conversion des formats demandés (Étape 5)
  # Prix : suppression du symbole £ et conversion en float
  prix_gbp = float(row["prix"].replace("£", ""))

  # Note : conversion textuelle ('One' -> 1, 'Two' -> 2, etc.)
  mapping_notes = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
  note_int = mapping_notes.get(row["note"], 0)

  data_niveau_2.append({
      "titre": row["titre"],
      "catégorie": categorie,
      "prix_gbp": prix_gbp,
      "note": note_int,
      "stock": stock,
      "upc": upc,
      "url": url_livre,
  })

  # Temporisation éthique d'une seconde par livre
  time.sleep(1)

# Création du DataFrame final nettoyé (Étape 5)
df_final = pd.DataFrame(data_niveau_2)
print("\nCollecte du second niveau terminée !")
print(df_final.head())

Début de la collecte du second niveau (fiches produits)...
[1/32] Scraping de : Sharp Objects...
[2/32] Scraping de : In a Dark, Dark Wood...
[3/32] Scraping de : The Past Never Ends...
[4/32] Scraping de : A Murder in Time...
[5/32] Scraping de : The Murder of Roger Ackroyd (H...
[6/32] Scraping de : The Last Mile (Amos Decker #2)...
[7/32] Scraping de : That Darkness (Gardiner and Re...
[8/32] Scraping de : Tastes Like Fear (DI Marnie Ro...
[9/32] Scraping de : A Time of Torment (Charlie Par...
[10/32] Scraping de : A Study in Scarlet (Sherlock H...
[11/32] Scraping de : Poisonous (Max Revere Novels #...
[12/32] Scraping de : Murder at the 42nd Street Libr...
[13/32] Scraping de : Most Wanted...
[14/32] Scraping de : Hide Away (Eve Duncan #20)...
[15/32] Scraping de : Boar Island (Anna Pigeon #19)...
[16/32] Scraping de : The Widow...
[17/32] Scraping de : Playing with Fire...
[18/32] Scraping de : What Happened on Beale Street ...
[19/32] Scraping de : The Bachelor Girl's Guide to M

In [19]:
# 1. Combien de livres, et quel stock total ?
nb_livres = len(df_final)
stock_total = df_final["stock"].sum()
print(f"Nombre de livres : {nb_livres} | Stock total : {stock_total}")

# 2. Quel est le prix moyen, et le livre le plus cher ?
prix_moyen = df_final["prix_gbp"].mean()
livre_plus_cher = df_final.loc[df_final["prix_gbp"].idxmax()]
print(f"Prix moyen : £{prix_moyen:.2f}")
print(
    f"Livre le plus cher : '{livre_plus_cher['titre']}' (£"
    f"{livre_plus_cher['prix_gbp']})"
)

# 3. Comment se répartissent les notes de 1 à 5 ?
repartition_notes = df_final["note"].value_counts().sort_index()
print("\nRépartition des notes (1 à 5) :")
print(repartition_notes)

# 4. La colonne catégorie contient-elle une seule valeur ?
valeurs_categories = df_final["catégorie"].unique()
print(f"\nCatégories uniques trouvées : {valeurs_categories}")

Nombre de livres : 32 | Stock total : 290
Prix moyen : £31.72
Livre le plus cher : 'Boar Island (Anna Pigeon #19)' (£59.48)

Répartition des notes (1 à 5) :
note
1    7
2    5
3    8
4    7
5    5
Name: count, dtype: int64

Catégories uniques trouvées : <StringArray>
['Mystery']
Length: 1, dtype: str


In [20]:
import os

os.makedirs("../data", exist_ok=True)
df_final.to_csv("../data/mystery.csv", index=False)
print("Fichier CSV enregistré avec succès dans ../data/")

Fichier CSV enregistré avec succès dans ../data/
